In [1]:
!pip install -U scikit-learn==1.7.2 xgboost==3.2.0 joblib==1.5.3 shap==0.48.0 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.4 MB/s eta 0:00:00


In [3]:
# =========================
# 1. Imports
# =========================
import numpy as np
import pandas as pd
import joblib
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from xgboost import XGBRegressor

warnings.filterwarnings("ignore")


# =========================
# 2. Load dataset
# =========================
data_path = "anuradhapura_corn_yield_dataset.csv"
df = pd.read_csv(data_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())


# =========================
# 3. Keep only columns used by the app
# =========================
required_columns = [
    "district",
    "variety",
    "soil_type",
    "irrigation_type",
    "pest_disease_level",
    "farm_size_acres",
    "seasonal_rainfall_mm",
    "fertilizer_kg_per_acre",
    "previous_yield_kg_per_acre",
    "yield_kg_per_acre"
]

missing_cols = [c for c in required_columns if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in dataset: {missing_cols}")

df = df[required_columns].copy()


# =========================
# 4. Clean categorical values
# =========================
cat_cols = [
    "district",
    "variety",
    "soil_type",
    "irrigation_type",
    "pest_disease_level"
]

for col in cat_cols:
    df[col] = df[col].astype(str).str.strip()

df["district"] = df["district"].replace({
    "anuradhapura": "Anuradhapura",
    "ANURADHAPURA": "Anuradhapura"
})

df["variety"] = df["variety"].replace({
    "Hybrid A": "Hybrid_A",
    "Hybrid B": "Hybrid_B",
    "OPV Local": "OPV_Local",
    "OPV local": "OPV_Local"
})

df["soil_type"] = df["soil_type"].replace({
    "Loamy": "Loam",
    "loam": "Loam",
    "clay": "Clay",
    "sandy": "Sandy"
})

df["irrigation_type"] = df["irrigation_type"].replace({
    "Tubewell": "Tube well",
    "Tube_well": "Tube well"
})

df["pest_disease_level"] = df["pest_disease_level"].replace({
    0: "None",
    1: "Low",
    2: "Medium",
    3: "High",
    "0": "None",
    "1": "Low",
    "2": "Medium",
    "3": "High"
})

print("\nUnique category values after cleaning:")
for c in cat_cols:
    print(f"{c}: {sorted(df[c].dropna().unique())}")


# =========================
# 5. Convert numeric columns
# =========================
num_cols = [
    "farm_size_acres",
    "seasonal_rainfall_mm",
    "fertilizer_kg_per_acre",
    "previous_yield_kg_per_acre"
]

target_col = "yield_kg_per_acre"

for col in num_cols + [target_col]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nMissing values before fill:")
print(df.isnull().sum())


# =========================
# 6. Basic outlier clipping (optional but useful)
# =========================
def clip_outliers_iqr(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return series.clip(lower, upper)

for col in num_cols + [target_col]:
    df[col] = clip_outliers_iqr(df[col])

print("\nTarget summary:")
print(df[target_col].describe())


# =========================
# 7. Train-test split
# =========================
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)


# =========================
# 8. Preprocessor
# =========================
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, cat_cols),
        ("num", numeric_transformer, num_cols),
    ]
)


# =========================
# 9. Candidate models
# =========================
models = {
    "RandomForest": RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),
    "XGBoost": XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=42
    )
}


# =========================
# 10. Train + evaluate
# =========================
results = []
best_name = None
best_model = None
best_rmse = float("inf")
best_pipeline = None

cv = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds) ** 0.5
    r2 = r2_score(y_test, preds)

    cv_scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="r2",
        n_jobs=-1
    )

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2_test": r2,
        "CV_R2_mean": cv_scores.mean(),
        "CV_R2_std": cv_scores.std()
    })

    print(f"\n{name}")
    print(f"MAE      : {mae:.2f}")
    print(f"RMSE     : {rmse:.2f}")
    print(f"R2 Test  : {r2:.4f}")
    print(f"CV R2 Avg: {cv_scores.mean():.4f} \u00b1 {cv_scores.std():.4f}")

    if rmse < best_rmse:
        best_rmse = rmse
        best_name = name
        best_model = model
        best_pipeline = pipeline

results_df = pd.DataFrame(results).sort_values(by="RMSE")
print("\nModel comparison:")
print(results_df)


# =========================
# 11. Save best model
# =========================
model_path = "corn_yield_model_best.pkl"
joblib.dump(best_pipeline, model_path)

print(f"\nBest model: {best_name}")
print(f"Saved to: {model_path}")


# =========================
# 12. Test one sample from test set
# =========================
sample_idx = X_test.index[0]
sample_input = X_test.loc[[sample_idx]]
actual_yield = y_test.loc[sample_idx]
predicted_yield = best_pipeline.predict(sample_input)[0]

print("\nSample input:")
print(sample_input)

print(f"\nActual yield   : {actual_yield:.2f}")
print(f"Predicted yield: {predicted_yield:.2f}")


# =========================
# 13. Optional: feature importance for tree model
# =========================
trained_model = best_pipeline.named_steps["model"]
ohe = best_pipeline.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]

encoded_cat_names = ohe.get_feature_names_out(cat_cols)
feature_names = list(encoded_cat_names) + num_cols

if hasattr(trained_model, "feature_importances_"):
    importances = trained_model.feature_importances_
    fi = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    }).sort_values("importance", ascending=False)

    print("\nTop 15 feature importances:")
    print(fi.head(15))

Shape: (1500, 10)

Columns:
['district', 'farm_size_acres', 'variety', 'soil_type', 'irrigation_type', 'seasonal_rainfall_mm', 'fertilizer_kg_per_acre', 'previous_yield_kg_per_acre', 'pest_disease_level', 'yield_kg_per_acre']

First 5 rows:
       district  farm_size_acres    variety soil_type irrigation_type  \
0  Anuradhapura            0.633  OPV_Local      Loam         Rainfed   
1  Anuradhapura            1.925  OPV_Local      Loam         Rainfed   
2  Anuradhapura            1.758  OPV_Local     Sandy           Canal   
3  Anuradhapura            0.717   Hybrid_A      Loam         Rainfed   
4  Anuradhapura            2.687   Hybrid_A     Sandy         Rainfed   

   seasonal_rainfall_mm  fertilizer_kg_per_acre  previous_yield_kg_per_acre  \
0                 858.5                   165.2                       550.8   
1                 873.7                   201.9                       968.1   
2                 222.8                   128.6                       817.9   
3   

In [4]:
import joblib
import pandas as pd

model = joblib.load("corn_yield_model_best.pkl")

sample = pd.DataFrame([{
    "district": "Anuradhapura",
    "variety": "OPV_Local",
    "soil_type": "Loam",
    "irrigation_type": "Rainfed",
    "pest_disease_level": "High",
    "farm_size_acres": 0.509,
    "seasonal_rainfall_mm": 1009.864,
    "fertilizer_kg_per_acre": 130.212,
    "previous_yield_kg_per_acre": 1818.143
}])

pred = model.predict(sample)
print("Predicted yield:", pred[0])

Predicted yield: 722.8595


In [5]:
import sklearn, xgboost, pandas, numpy, joblib
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgboost.__version__)
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("joblib:", joblib.__version__)

scikit-learn: 1.7.2
xgboost: 3.2.0
pandas: 2.2.2
numpy: 2.0.2
joblib: 1.5.3
